# English-Domain Input Samples Demo

Browse 300 cached English samples across six domains: 50 per domain, each with 2,000 Llama-3.2-3B content tokens.

Sources (training splits):

| Domain | Dataset and subset |
| --- | --- |
| News | [CNN/DailyMail](https://huggingface.co/datasets/abisee/cnn_dailymail), `3.0.0` |
| Fiction | [Project Gutenberg](https://huggingface.co/datasets/Despina/project_gutenberg), `fiction_books_in_chunks` |
| Wikipedia | [Wikimedia Wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia), `20231101.en` |
| arXiv | [arXiv Summarization](https://huggingface.co/datasets/ccdv/arxiv-summarization), `document` |
| Dialogue | [DailyDialog](https://huggingface.co/datasets/OpenRL/daily_dialog) |
| Legal | [LexGLUE](https://huggingface.co/datasets/coastalcph/lex_glue), `ecthr_a` |

Light cleanup normalized whitespace and control characters, decoded HTML entities, removed selected boilerplate, and corrected dialogue punctuation spacing.

Run all cells from `English-domain-inputs` to browse cached text, token IDs, and metadata in `natural_language_samples`. Diagnostics are computed by `Generate-english-domain-spectral-diagnostics.ipynb`.


In [1]:
from pathlib import Path
import html
import json

from IPython.display import HTML, clear_output, display

DATA_DIR = Path("natural_language_samples")
SAMPLE_JSONL_PATH = DATA_DIR / "natural_language_2000_token_windows.jsonl"

if not SAMPLE_JSONL_PATH.is_file():
    raise FileNotFoundError(
        f"Cached sample file not found: {SAMPLE_JSONL_PATH.name}. "
        "Run this notebook from English-domain-inputs, or set DATA_DIR to the "
        "folder containing natural_language_2000_token_windows.jsonl. "
        "If the file is missing, restore it from the shared sample data."
    )

with SAMPLE_JSONL_PATH.open(encoding="utf-8") as sample_file:
    sample_browser_records = [json.loads(line) for line in sample_file if line.strip()]

if not sample_browser_records:
    raise ValueError(f"The cached sample file is empty: {SAMPLE_JSONL_PATH}")

sample_browser_by_family = {}
sample_browser_by_id = {}
for record in sample_browser_records:
    required_fields = {"family", "sample_id", "text", "token_ids", "num_tokens"}
    missing_fields = required_fields.difference(record)
    if missing_fields:
        raise ValueError(f"A cached sample is missing fields: {sorted(missing_fields)}")
    sample_id = record["sample_id"]
    if sample_id in sample_browser_by_id:
        raise ValueError(f"Duplicate cached sample ID: {sample_id}")
    if len(record["token_ids"]) != record["num_tokens"]:
        raise ValueError(f"Token count does not match the saved token IDs for {sample_id}.")
    sample_browser_by_family.setdefault(record["family"], []).append(record)
    sample_browser_by_id[sample_id] = record

for records in sample_browser_by_family.values():
    records.sort(key=lambda record: int(record.get("run_id", 0)))

DOMAIN_LABELS = {
    "news": "News",
    "fiction": "Fiction",
    "wiki": "Wikipedia",
    "arxiv": "arXiv articles",
    "dialogue": "Dialogue",
    "legal": "Legal text",
}

print(
    f"Loaded {len(sample_browser_records):,} cached samples "
    f"across {len(sample_browser_by_family)} domains."
)

Loaded 300 cached samples across 6 domains.


Browse all cached samples by domain and sample ID. **Text** shows the saved text, **Token IDs** shows its stored content-token sequence, and **Metadata** shows available source details.

In [2]:
BROWSER_CSS = """
<style>
.nl-browser-card {
    border: 1px solid #d0d7de;
    border-radius: 6px;
    padding: 14px 16px;
    background: #ffffff;
    color: #24292f;
    max-width: 1180px;
}
.nl-browser-title {
    font: 600 15px/1.35 -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    margin-bottom: 4px;
}
.nl-browser-subtitle {
    color: #57606a;
    font: 13px/1.35 -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    margin-bottom: 12px;
}
.nl-browser-text {
    max-height: 620px;
    overflow-y: auto;
    white-space: pre-wrap;
    overflow-wrap: anywhere;
    font: 15px/1.58 Georgia, "Times New Roman", serif;
    padding-right: 8px;
}
.nl-browser-tokenids {
    max-height: 620px;
    overflow: auto;
    white-space: pre;
    font: 12px/1.55 ui-monospace, SFMono-Regular, Consolas, "Liberation Mono", monospace;
    background: #f6f8fa;
    border: 1px solid #d8dee4;
    border-radius: 6px;
    padding: 12px;
}
.nl-browser-meta {
    border-collapse: collapse;
    font: 13px/1.45 -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
}
.nl-browser-meta th, .nl-browser-meta td {
    border-bottom: 1px solid #d8dee4;
    padding: 7px 10px;
    text-align: left;
    vertical-align: top;
}
.nl-browser-meta th {
    color: #57606a;
    font-weight: 600;
    white-space: nowrap;
}
</style>
"""

def token_id_lines(token_ids, per_line=25):
    lines = []
    for start in range(0, len(token_ids), per_line):
        end = min(start + per_line, len(token_ids))
        values = " ".join(str(token_id) for token_id in token_ids[start:end])
        lines.append(f"{start:04d}-{end - 1:04d}: {values}")
    return "\n".join(lines)


def metadata_table(record):
    rows = [
        ("Domain", DOMAIN_LABELS.get(record["family"], record["family"])),
        ("Sample", record["sample_id"]),
        ("Content tokens", record["num_tokens"]),
        ("Tokenizer", record.get("tokenizer_name")),
        ("Title", record.get("source_title") or record.get("fiction_title")),
        ("Author", record.get("source_author") or record.get("fiction_author")),
        ("Year", record.get("source_year") or record.get("fiction_year")),
        ("Release date", record.get("source_release_date") or record.get("fiction_release_date")),
        ("Source IDs", record.get("source_ids") or record.get("source_id")),
    ]
    body = ""
    for label, value in rows:
        if value is None or value == "" or value == []:
            continue
        if isinstance(value, list):
            value = ", ".join(str(item) for item in value)
        body += (
            f"<tr><th>{html.escape(label)}</th>"
            f"<td>{html.escape(str(value))}</td></tr>"
        )
    return f"<table class='nl-browser-meta'>{body}</table>"


def render_record(record, view="Text", preview_chars=None):
    domain = DOMAIN_LABELS.get(record["family"], record["family"])
    title = html.escape(
        f"{record['sample_id']} | {domain} | {record['num_tokens']:,} content tokens"
    )
    source_parts = [
        record.get("source_title") or record.get("fiction_title"),
        record.get("source_author") or record.get("fiction_author"),
        record.get("source_year") or record.get("fiction_year"),
    ]
    subtitle = " | ".join(str(part) for part in source_parts if part)
    if view == "Token IDs":
        body = f"<pre class='nl-browser-tokenids'>{html.escape(token_id_lines(record['token_ids']))}</pre>"
    elif view == "Metadata":
        body = metadata_table(record)
    elif view == "Text":
        text = record["text"]
        if preview_chars is not None and len(text) > preview_chars:
            text = text[:preview_chars].rstrip() + "…"
            subtitle = " | ".join(filter(None, [subtitle, "Excerpt; open the browser for the full sample."]))
        body = f"<div class='nl-browser-text'>{html.escape(text)}</div>"
    else:
        raise ValueError(f"Unknown sample view: {view}")
    return HTML(
        BROWSER_CSS
        + "<div class='nl-browser-card'>"
        + f"<div class='nl-browser-title'>{title}</div>"
        + f"<div class='nl-browser-subtitle'>{html.escape(subtitle)}</div>"
        + body
        + "</div>"
    )


try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
    print("Interactive controls are unavailable. The static previews below still work.")

if widgets is not None:
    browser_style = {"description_width": "65px"}
    family_dropdown = widgets.Dropdown(
        options=[(DOMAIN_LABELS.get(family, family), family) for family in sample_browser_by_family],
        description="Domain:",
        style=browser_style,
        layout=widgets.Layout(width="260px"),
    )
    sample_dropdown = widgets.Dropdown(
        options=[],
        description="Sample:",
        style=browser_style,
        layout=widgets.Layout(width="260px"),
    )
    view_toggle = widgets.ToggleButtons(
        options=["Text", "Token IDs", "Metadata"],
        value="Text",
        description="View:",
        style=browser_style,
    )
    browser_output = widgets.Output(layout=widgets.Layout(width="100%"))

    def render_current(*_):
        with browser_output:
            clear_output(wait=True)
            if sample_dropdown.value is not None:
                display(render_record(sample_browser_by_id[sample_dropdown.value], view_toggle.value))

    def update_sample_options(*_):
        records = sample_browser_by_family[family_dropdown.value]
        sample_dropdown.options = [record["sample_id"] for record in records]
        sample_dropdown.value = records[0]["sample_id"]
        render_current()

    family_dropdown.observe(update_sample_options, names="value")
    sample_dropdown.observe(render_current, names="value")
    view_toggle.observe(render_current, names="value")

    update_sample_options()
    sample_browser = widgets.VBox([
        widgets.HBox(
            [family_dropdown, sample_dropdown, view_toggle],
            layout=widgets.Layout(flex_flow="row wrap"),
        ),
        browser_output,
    ])
    display(sample_browser)

Static previews show the first 700 characters of one sample per domain. Change `PREVIEW_SAMPLE_INDEX` to view a different sample, or set `PREVIEW_CHARS = None` to show the full text.

In [3]:
PREVIEW_CHARS = 700
PREVIEW_SAMPLE_INDEX = 1

for family, records in sample_browser_by_family.items():
    if not 0 <= PREVIEW_SAMPLE_INDEX < len(records):
        raise ValueError(
            f"PREVIEW_SAMPLE_INDEX must be between 0 and {len(records) - 1} for {family}."
        )
    display(render_record(records[PREVIEW_SAMPLE_INDEX], preview_chars=PREVIEW_CHARS))